In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('dataset/diabetes.csv')
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [6]:
df.isna().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [21]:
X = df.iloc[:,:-1]
y = df.iloc[:,-1]

X

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
0,6,148,72,35,0,33.6,0.627,50
1,1,85,66,29,0,26.6,0.351,31
2,8,183,64,0,0,23.3,0.672,32
3,1,89,66,23,94,28.1,0.167,21
4,0,137,40,35,168,43.1,2.288,33
...,...,...,...,...,...,...,...,...
763,10,101,76,48,180,32.9,0.171,63
764,2,122,70,27,0,36.8,0.340,27
765,5,121,72,23,112,26.2,0.245,30
766,1,126,60,0,0,30.1,0.349,47


In [22]:
y

0      1
1      0
2      1
3      0
4      1
      ..
763    0
764    0
765    0
766    1
767    0
Name: Outcome, Length: 768, dtype: int64

In [23]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [25]:
X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=.2,random_state=4)
X_train

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age
596,0,67,76,0,0,45.3,0.194,46
90,1,80,55,0,0,19.1,0.258,21
734,2,105,75,0,0,23.3,0.560,53
694,2,90,60,0,0,23.5,0.191,25
517,7,125,86,0,0,37.6,0.304,51
...,...,...,...,...,...,...,...,...
360,5,189,64,33,325,31.2,0.583,29
709,2,93,64,32,160,38.0,0.674,23
439,6,107,88,0,0,36.8,0.727,31
174,2,75,64,24,55,29.7,0.370,33


In [10]:
y_train

596    0
90     0
734    0
694    0
517    0
      ..
360    1
709    1
439    0
174    0
122    0
Name: Outcome, Length: 614, dtype: int64

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier,VotingClassifier
from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import cross_val_score, GridSearchCV

In [27]:
scale = StandardScaler()
X_train_scaled = scale.fit_transform(X_train)
X_test_scaled = scale.transform(X_test)

In [29]:
lr = LogisticRegression()
lr.fit(X_train_scaled,y_train)
y_pred1 = lr.predict(X_test_scaled)

print(accuracy_score(y_pred1,y_test))
score = cross_val_score(LogisticRegression(), X_train_scaled,y_train,cv=10)
print(score.mean())

0.8116883116883117
0.7620306716023267


In [ ]:
models = {
    "dt": DecisionTreeClassifier(),
    "rf": RandomForestClassifier(),
    "gb": GradientBoostingClassifier(),
    "xgb": XGBClassifier(eval_metric='logloss')
}

for name, model in models.items():
    score = cross_val_score(model, X_train,y_train,cv=10)
    print(f"Name: {name}, accuracy: {score.mean():.4f}")

Name: lg, accuracy: 0.7620
Name: dt, accuracy: 0.6727
Name: rf, accuracy: 0.7687
Name: gb, accuracy: 0.7491
Name: xgb, accuracy: 0.7328


In [ ]:
vt = VotingClassifier(
    estimators=[
        ('rf1',RandomForestClassifier(random_state=4)),
        ('gb1',GradientBoostingClassifier()),
        ('xgb1',XGBClassifier(eval_metric='logloss')),
        ('dt1',DecisionTreeClassifier(random_state=4))
    ],
    voting='hard'
)

score = cross_val_score(vt,X_train,y_train,cv=10)
score.mean()

np.float64(0.7523003701745108)

In [ ]:
best_model = RandomForestClassifier(random_state=4)

In [16]:
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 5, 10, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2'],
    'bootstrap': [True, False]
}

grid = GridSearchCV(
    best_model,
    param_grid,
    cv=10,
    n_jobs=-1
)

grid.fit(X_train,y_train)

print(grid.best_params_)

best_model = grid.best_estimator_

{'bootstrap': True, 'max_depth': 20, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [18]:
print(grid.best_score_)

0.7865415124272872


In [17]:
y_pred = best_model.predict(X_test)

print(accuracy_score(y_test,y_pred))
print(confusion_matrix(y_test,y_pred))
print(classification_report(y_test,y_pred))

0.7727272727272727
[[88 14]
 [21 31]]
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       102
           1       0.69      0.60      0.64        52

    accuracy                           0.77       154
   macro avg       0.75      0.73      0.74       154
weighted avg       0.77      0.77      0.77       154



In [30]:
import pickle

filename = 'trained_model.sav'
pickle.dump(best_model,open(filename,'wb'))